In [10]:
import pandas as pd, numpy as np, pyarrow.parquet as pq
BASE = "/scratch/sm3664/medicare-fraud/processed/part_b/labeled"
for f in ["part_b_labeled_2019_2023.parquet", "leie_cumulative_2019_2023.parquet"]:
    pf = pq.ParquetFile(f"{BASE}/{f}")
    print(f, pf.metadata.num_rows, "rows")
    print([c for c in pf.schema_arrow.names if not c.startswith(("Tot_", "Avg_", "Bene_", "Drug_", "Med_"))], "\n")

part_b_labeled_2019_2023.parquet 5626423 rows
['Rndrng_NPI', 'year', 'Rndrng_Prvdr_Type', 'Rndrng_Prvdr_Ent_Cd', 'fraud'] 

leie_cumulative_2019_2023.parquet 7970 rows
['NPI', 'EXCLTYPE', 'EXCLDATE', 'snapshot_year', 'last_excl_year'] 



In [11]:
LABEL, NPI, YEAR = "fraud", "Rndrng_NPI", "year"
LEIE_NPI, LEIE_DATE = "NPI", "EXCLDATE"

In [12]:
df = pd.read_parquet(f"{BASE}/part_b_labeled_2019_2023.parquet", columns=[NPI, YEAR, LABEL])
t = df.groupby(YEAR).agg(rows=(LABEL, "size"), fraud_rows=(LABEL, "sum"))
t["fraud_rate_%"] = (100 * t.fraud_rows / t.rows).round(4)
t["fraud_npis"] = df[df[LABEL] == 1].groupby(YEAR)[NPI].nunique()
print(t)

         rows  fraud_rows  fraud_rate_%  fraud_npis
year                                               
2019  1093367         840        0.0768         840
2020  1085313         671        0.0618         671
2021  1123589         529        0.0471         529
2022  1148873         369        0.0321         369
2023  1175281         243        0.0207         243


In [13]:
leie = pd.read_parquet(f"{BASE}/leie_cumulative_2019_2023.parquet", columns=[LEIE_NPI, LEIE_DATE, "last_excl_year"])
key = lambda s: pd.to_numeric(s, errors="coerce").astype("Int64").astype(str)
leie["k"] = key(leie[LEIE_NPI])
fr = df[df[LABEL] == 1].copy()
fr["k"] = key(fr[NPI])
fr["lag"] = fr.k.map(leie.groupby("k")[LEIE_DATE].max().dt.year) - fr[YEAR]
fr["lag_label"] = fr.k.map(leie.groupby("k")["last_excl_year"].max()) - fr[YEAR]
print("no match:", fr.lag.isna().sum())
print("Years from billing to exclusion date:\n", pd.crosstab(fr[YEAR], fr.lag.clip(upper=5), margins=True))
print("Years from billing to last_excl_year:\n", pd.crosstab(fr[YEAR], fr.lag_label.clip(upper=5), margins=True))

no match: 0
Years from billing to exclusion date:
 lag   -4  -3   0    1    2    3    4    5   All
year                                           
2019   1   1   5   32   46  123  131  501   840
2020   1   0   6   16   81  104  171  292   671
2021   0   0   3   39   71  137  132  147   529
2022   0   0   5   38   87  104  135    0   369
2023   0   0   7   40   77  119    0    0   243
All    2   1  26  165  362  587  569  940  2652
Years from billing to last_excl_year:
 lag_label   1    2    3    4     5   All
year                                    
2019        6   21   22   61   730   840
2020        8   12   49   49   553   671
2021        6   28   37   88   370   529
2022        6   22   66   56   219   369
2023        7   41   47   50    98   243
All        33  124  221  304  1970  2652
